# 03 Gaze Shift Heatmaps

This notebook runs the grating gaze grid and visualizes where inferred PO changes are largest. Heat maps are useful because gaze shifts have both magnitude and direction. The debug grid is coarse; the full config uses 0.1 degree steps from 0 to 10 degrees.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.utils import load_config, rng_from_config
from src.sampling import sample_population
from src.simulation import run_grating_shift_grid
from src.plotting import set_plot_style, heatmap_from_table

set_plot_style()
fig_dir = ROOT / 'results/figures'
fig_dir.mkdir(parents=True, exist_ok=True)

cfg = load_config(ROOT / 'configs/default.yaml', debug=True)
rng = rng_from_config(cfg)
pop = sample_population(cfg, rng)
result = run_grating_shift_grid(pop, cfg, rng)
summary = result['summary']
display(summary.head())

## Primary Heat Maps

The first heat map asks whether the inferred PO moves. The next two ask whether the tuning curve changes without necessarily rotating: circular variance can rise, and vector strength can fall.

In [ ]:
heatmap_from_table(summary, 'median_abs_delta_po_deg', fig_dir / 'notebook03_median_abs_delta_po.png', title='Median |ΔPO|')
heatmap_from_table(summary, 'median_circular_variance_change', fig_dir / 'notebook03_cv_change.png', title='Median circular variance change', cmap='vlag', center=0)
heatmap_from_table(summary, 'median_tuning_strength_change', fig_dir / 'notebook03_strength_change.png', title='Median tuning strength change', cmap='vlag', center=0)
display(summary.tail())

## Neuron-Level Sensitivity at the Largest Offset

A population median can hide heterogeneity. The next plots ask whether sensitive neurons are associated with RF size or preferred SF. Narrower or higher-SF RFs are expected to be more phase-sensitive.

In [ ]:
far = result['far_state']
changes = far['neuron_changes']
rf_size = 0.5 * (pop.sigma_x_deg.to_numpy() + pop.sigma_y_deg.to_numpy())
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
sns.histplot(changes['abs_delta_po_deg'], bins=24, ax=axes[0], color='#4C78A8')
axes[0].set_xlabel('|ΔPO| (deg)'); axes[0].set_title('Distribution at largest offset')
axes[1].scatter(rf_size, changes['abs_delta_po_deg'], s=16, alpha=0.7)
axes[1].set_xlabel('Mean RF sigma (deg)'); axes[1].set_ylabel('|ΔPO| (deg)'); axes[1].set_title('RF size')
axes[2].scatter(pop.f0_cpd, changes['abs_delta_po_deg'], s=16, alpha=0.7)
axes[2].set_xlabel('Preferred SF (cpd)'); axes[2].set_ylabel('|ΔPO| (deg)'); axes[2].set_title('Preferred SF')
plt.tight_layout()
plt.savefig(fig_dir / 'notebook03_neuron_sensitivity_summary.png', bbox_inches='tight')

## Example Tuning Curves

These examples are selected by low, median, and high `|ΔPO|`. Some neurons rotate their estimated PO; others mainly flatten or reshape their tuning curve.

In [ ]:
selected = np.argsort(changes['abs_delta_po_deg'])
example_ids = [int(selected[0]), int(selected[len(selected)//2]), int(selected[-1])]
base_tuning = result['baseline_tuning']['tuning_curve']
shift_tuning = far['tuning']['tuning_curve']
oris = result['stimuli']['orientations_deg']
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=False)
for ax, neuron in zip(axes, example_ids):
    ax.plot(oris, base_tuning[neuron], 'o-', label='baseline')
    ax.plot(oris, shift_tuning[neuron], 's-', label='largest offset')
    ax.set_title(f"cell {neuron}, |ΔPO|={changes['abs_delta_po_deg'][neuron]:.1f} deg")
    ax.set_xlabel('Orientation (deg)'); ax.set_ylabel('Response')
axes[0].legend(frameon=False)
plt.tight_layout()
plt.savefig(fig_dir / 'notebook03_example_tuning_curves.png', bbox_inches='tight')

## Takeaway

Interpret grating heat maps together with phase-sanity checks. Large ΔPO in this simple-cell grating pipeline is not evidence that gaze physically rotates full-field gratings; it shows where phase-sensitive or noisy estimation can make the inferred PO unstable.